# Lecture 13: Dense and Sparse Vectors in Search

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~15 minutes | **Level:** Intermediate  

---

## Overview

In previous lectures, you learned about **dense vector search** using embeddings.  
Dense search is powerful for understanding meaning, but it has limitations.

In this lecture, you will learn:

1. **Why dense search alone is not enough** (with manual examples)
2. **What sparse vectors are** and when they excel
3. **How to create a dense-only vector store** in Qdrant
4. **How to create a hybrid vector store** (dense + sparse) in Qdrant
5. **When to use each approach**

> **After this lecture:** You will understand the difference between dense and sparse  
> vectors and know how to use both in production RAG systems.

---

## 1. Setup

Install required packages and set API keys.

In [1]:
# Install required packages
%pip install langchain langchain-qdrant langchain-huggingface langchain-community qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

# Qdrant Cloud credentials
QDRANT_URL = ""
QDRANT_API_KEY = ""

print("Qdrant credentials configured")

Qdrant credentials configured


---

## 2. Why Dense Search is Not Enough

### The Problem with Dense Vectors

Dense vectors (embeddings) capture **semantic meaning** but struggle with **exact matches**.

### Manual Example 1: Dense Search Captures Meaning

```python
# These queries have similar dense embeddings (similar meaning)
query1 = "heart attack symptoms"
query2 = "signs of myocardial infarction"
query3 = "cardiac arrest warning signs"
```

**Dense search understands:** These queries mean similar things.  
**Result:** All three find similar documents about heart problems.

### Manual Example 2: Dense Search Fails on Exact Matches

```python
# Looking for a specific chemical by exact name
query = "3-fluorophenmetrazine"

# Document 1: "The substance 3-fluorophenmetrazine is a stimulant"
# Document 2: "Fluorinated stimulant compounds include phenmetrazine derivatives"
# Document 3: "Synthetic cathinones and related stimulants"
```

**Dense search problem:** All three documents have similar embeddings because they discuss  
similar topics (stimulants, chemicals). Dense search might return Document 2 or 3 instead  
of Document 1, even though only Document 1 contains the **exact chemical name**.

### When Dense Search Struggles

| Query Type | Example | Dense Search Performance |
|-----------|---------|-------------------------|
| **Exact names** | "flubromazepam" | Poor - returns similar chemicals |
| **Product codes** | "Model XR-500" | Poor - returns similar products |
| **CAS numbers** | "439-14-5" | Poor - treats as random numbers |
| **IDs** | "Error E-404" | Poor - focuses on "error" concept |
| **Natural language** | "What causes headaches?" | Excellent - understands meaning |

### The Solution: Sparse Vectors

**Sparse vectors** (also called keyword or BM25 search) work like traditional search engines:

- They find documents containing **exact word matches**
- They count **how many times** keywords appear
- They score **rare words** higher (e.g., "3-fluorophenmetrazine" is rare, "the" is common)

### Manual Example 3: Sparse Search Excels on Exact Matches

```python
query = "3-fluorophenmetrazine"

# Sparse search looks for EXACT word match
# Document 1: "The substance 3-fluorophenmetrazine is a stimulant" -> MATCH! High score
# Document 2: "Fluorinated stimulant compounds include phenmetrazine" -> Partial match, lower score
# Document 3: "Synthetic cathinones and related stimulants" -> No match
```

**Sparse search returns Document 1 first** because it contains the exact keyword.

### The Best Solution: Use Both!

| Search Type | Strengths | Weaknesses |
|------------|-----------|------------|
| **Dense (embeddings)** | Understands meaning, handles synonyms | Misses exact matches |
| **Sparse (keywords)** | Finds exact words, codes, names | Doesn't understand meaning |
| **Hybrid (dense + sparse)** | Gets both meaning AND exact matches | None - best of both! |

> **Production RAG systems use hybrid search** to handle all query types.

---

## 3. Load WHO Document

We will use the WHO Substances Under Surveillance PDF.  
This document contains:

- **Exact chemical names** (e.g., "3-fluorophenmetrazine") - needs sparse search
- **Health descriptions** (e.g., "opioid-like effects") - needs dense search

**Download the PDF:**

1. URL: https://cdn.who.int/media/docs/default-source/controlled-substances/47th-ecdd/2024-who-substances-under-surveillance.pdf
2. Save to: `data/who_substances_surveillance.pdf`

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
pdf_path = "data/who_substances_surveillance.pdf"

if not os.path.exists(pdf_path):
    print(f"ERROR: PDF not found at {pdf_path}")
    print("Please download from: https://cdn.who.int/media/docs/default-source/controlled-substances/47th-ecdd/2024-who-substances-under-surveillance.pdf")
else:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    # Chunk the documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    
    print(f"Loaded {len(documents)} pages")
    print(f"Created {len(chunks)} chunks")
    print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

c:\Users\mhari\miniconda3\envs\demo1\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)


Loaded 14 pages
Created 53 chunks
Average chunk size: 701 characters


---

## 4. Create Dense-Only Vector Store

First, let's create a traditional vector store using only dense vectors.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

print("Creating dense-only vector store...")

# Create embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create dense-only vector store
dense_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="who_dense_only",
    force_recreate=True,
)

print(f"Dense-only collection created: who_dense_only")
print(f"  - Vector size: 384 dimensions (HuggingFace all-MiniLM-L6-v2)")
print(f"  - Documents indexed: {len(chunks)}")

Creating dense-only vector store...


c:\Users\mhari\miniconda3\envs\demo1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 456.53it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dense-only collection created: who_dense_only
  - Vector size: 384 dimensions (HuggingFace all-MiniLM-L6-v2)
  - Documents indexed: 53


### Test Dense-Only Search

Let's test with an exact chemical name query.

In [5]:
# Test 1: Exact chemical name (dense search will struggle)
query1 = "3-fluorophenmetrazine"
results1 = dense_store.similarity_search(query1, k=3)

print(f"Query: {query1}")
print(f"Expected: Should find the exact substance entry\n")

for i, doc in enumerate(results1):
    has_match = query1.lower() in doc.page_content.lower()
    # match_label = "[EXACT MATCH]" if has_match else "[No exact match]"
    # print(f"Result {i+1}: {match_label}")
    print(f"  {doc.page_content[:150]}...\n")

# Test 2: Semantic query (dense search should work)
query2 = "Which substances cause opioid-like effects?"
results2 = dense_store.similarity_search(query2, k=3)

print(f"\nQuery: {query2}")
print(f"Expected: Should find documents about opioid effects\n")

for i, doc in enumerate(results2):
    print(f"Result {i+1}:")
    print(f"  {doc.page_content[:150]}...\n")

Query: 3-fluorophenmetrazine
Expected: Should find the exact substance entry

  . 3-fluorophenmetrazine	(3-FPM)	Recommended for surveillance at 43rd ECDD (2020) Last review: Critical review in the 43rd ECDD (2020) 3-Fluorophenmetr...

  . 6 3-Ethylheptedrone (3-ethylnorheptedrone) ............................................................................... 6 4-Fluoromethcathinone (...

  Updated March 24, 2025 para-fluorofuranyl fentanyl (4-fluorofuranyl fentanyl) ................................................................ 8 Krato...


Query: Which substances cause opioid-like effects?
Expected: Should find documents about opioid effects

Result 1:
  . At present there is insufficient information on dependence, abuse and risks to public health associated with use of this substance to enable conside...

Result 2:
  .  Opioids	Ethyleneoxynitazene	Added to the surveillance list in the May 2024 Working Group Meeting  Last review: N/A Ethyleneoxynitazene is a 5-nitro...

Result 3:
 

---

## 5. Create Hybrid Vector Store (Dense + Sparse)

Now let's create a collection that uses **both** dense and sparse vectors.

Qdrant supports hybrid search natively by storing both vector types in the same collection.

In [6]:
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams

print("Creating hybrid vector store (dense + sparse)...")

# Initialize Qdrant client
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Delete collection if exists
try:
    client.delete_collection("who_hybrid")
except:
    pass

# Create collection with BOTH dense and sparse vectors
client.create_collection(
    collection_name="who_hybrid",
    vectors_config={
        "dense": VectorParams(
            size=384,  # HuggingFace embedding size
            distance=Distance.COSINE,
        )
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,
            )
        )
    },
)

print("Hybrid collection created: who_hybrid")
print("  - Dense vectors: 384 dimensions (cosine similarity)")
print("  - Sparse vectors: BM25-style keyword matching")
print("\nIndexing documents...")

# Helper function to create sparse vectors (BM25-style)
def create_sparse_vector(text):
    """Create a simple sparse vector from text using word frequency."""
    words = text.lower().split()
    word_counts = {}
    for word in words:
        word = word.strip('.,!?;:"\'')
        if len(word) > 2:  # Ignore very short words
            word_counts[word] = word_counts.get(word, 0) + 1

    # Convert to sparse vector format with UNIQUE indices
    # Use dict to handle hash collisions (sum values if indices collide)
    index_values = {}
    for word, count in word_counts.items():
        idx = hash(word) % 100000
        # If collision occurs, sum the values
        index_values[idx] = index_values.get(idx, 0.0) + float(count)

    # Convert to sorted lists (Qdrant requires unique indices)
    indices = sorted(index_values.keys())
    values = [index_values[idx] for idx in indices]

    return models.SparseVector(indices=indices, values=values)

# Index documents with both dense and sparse vectors
points = []
for i, chunk in enumerate(chunks):
    # Get dense embedding
    dense_vector = embeddings.embed_query(chunk.page_content)

    # Create sparse vector
    sparse_vector = create_sparse_vector(chunk.page_content)

    # Create point with both vectors
    point = models.PointStruct(
        id=i,
        vector={
            "dense": dense_vector,
            "sparse": sparse_vector,
        },
        payload={
            "text": chunk.page_content,
            "source": chunk.metadata.get("source", ""),
        },
    )
    points.append(point)

# Upload points
client.upsert(collection_name="who_hybrid", points=points)

print(f"Indexed {len(points)} documents with dense + sparse vectors")

Creating hybrid vector store (dense + sparse)...
Hybrid collection created: who_hybrid
  - Dense vectors: 384 dimensions (cosine similarity)
  - Sparse vectors: BM25-style keyword matching

Indexing documents...
Indexed 53 documents with dense + sparse vectors


### Test Hybrid Search

Now let's search using both dense and sparse vectors together.

In [7]:
# Test 1: Exact chemical name (sparse vectors should help!)
query1 = "3-fluorophenmetrazine"

# Get dense embedding for query
query_dense = embeddings.embed_query(query1)

# Create sparse vector for query
query_sparse = create_sparse_vector(query1)

print(f"Query: {query1}")
print(f"Expected: Hybrid search should find the exact match\n")

# Search using dense vectors
results_dense = client.query_points(
    collection_name="who_hybrid",
    query=query_dense,
    using="dense",
    limit=6,
)

print("Dense search results:")
for i, hit in enumerate(results_dense.points):
    text = hit.payload["text"]
    has_match = query1.lower() in text.lower()
    match_label = "[EXACT MATCH]" if has_match else "[No exact match]"
    print(f"  {i+1}. {match_label}: {text[:150]}...")

# Search using sparse vectors
results_sparse = client.query_points(
    collection_name="who_hybrid",
    query=query_sparse,
    using="sparse",
    limit=3,
)

print("\nSparse search results:")
for i, hit in enumerate(results_sparse.points):
    text = hit.payload["text"]
    has_match = query1.lower() in text.lower()
    match_label = "[EXACT MATCH]" if has_match else "[No exact match]"
    print(f"  {i+1}. {match_label}: {text[:150]}...")

Query: 3-fluorophenmetrazine
Expected: Hybrid search should find the exact match

Dense search results:
  1. [EXACT MATCH]: . 3-fluorophenmetrazine	(3-FPM)	Recommended for surveillance at 43rd ECDD (2020) Last review: Critical review in the 43rd ECDD (2020) 3-Fluorophenmetr...
  2. [EXACT MATCH]: . 6 3-Ethylheptedrone (3-ethylnorheptedrone) ............................................................................... 6 4-Fluoromethcathinone (...
  3. [No exact match]: Updated March 24, 2025 para-fluorofuranyl fentanyl (4-fluorofuranyl fentanyl) ................................................................ 8 Krato...
  4. [No exact match]: . 7 Para-methoxy-butyrylfentanyl ................................................................................................... 8...
  5. [No exact match]: . 10 3-hydroxyphencyclidine (3-OH-PCP) ......................................................................................... 10 PSYCHOACTIVE MEDIC...
  6. [No exact match]: . It has n

In [8]:
# Test 2: Semantic query (both should work, dense might be better)
query2 = "Which substances cause opioid-like effects?"

# Get dense embedding for query
query_dense2 = embeddings.embed_query(query2)

# Create sparse vector for query
query_sparse2 = create_sparse_vector(query2)

print(f"Query: {query2}")
print(f"Expected: Dense search understands meaning better\n")

# Search using dense vectors
results_dense2 = client.query_points(
    collection_name="who_hybrid",
    query=query_dense2,
    using="dense",
    limit=3,
)

print("Dense search results:")
for i, hit in enumerate(results_dense2.points):
    print(f"  {i+1}. {hit.payload['text'][:150]}...")

# Search using sparse vectors
results_sparse2 = client.query_points(
    collection_name="who_hybrid",
    query=query_sparse2,
    using="sparse",
    limit=3,
)

print("\nSparse search results:")
for i, hit in enumerate(results_sparse2.points):
    print(f"  {i+1}. {hit.payload['text'][:150]}...")

Query: Which substances cause opioid-like effects?
Expected: Dense search understands meaning better

Dense search results:
  1. . At present there is insufficient information on dependence, abuse and risks to public health associated with use of this substance to enable conside...
  2. .  Opioids	Ethyleneoxynitazene	Added to the surveillance list in the May 2024 Working Group Meeting  Last review: N/A Ethyleneoxynitazene is a 5-nitro...
  3. LIST OF SUBSTANCES UNDER SURVEILLANCE 2024 A substance is placed on WHO ECDDs Surveillance List if the Committee considers the evidence of the impact ...

Sparse search results:
  1. Benzodiazepines	(NPS)	Adinazolam	Added to surveillance list by 9th WG meeting (2021) Last review: Critical review 44th ECDD (2022)   Adinazolam (IUPAC...
  2. JWH-073		Recommended for surveillance at 36th ECDD (2014) Last review: Critical review 38th ECDD (2016) Information has been brought to WHO’s attentio...
  3. . At present there is insufficient information on de

---

## 6. LangChain Hybrid Search (EnsembleRetriever)

Now let's use **LangChain** to combine dense and sparse search automatically.

### What is EnsembleRetriever?

`EnsembleRetriever` is LangChain's built-in way to combine multiple retrievers:
- Uses **Reciprocal Rank Fusion (RRF)** algorithm to merge results
- Combines rankings (not scores) to avoid scale issues
- Production-ready and battle-tested

### How RRF Works

Instead of combining different similarity scores, RRF combines the **rank positions**:

```
RRF_score(doc) = weight1/(k + rank_in_retriever1) + weight2/(k + rank_in_retriever2)
```

Where `k=60` is a constant that prevents top documents from dominating too much.

### Why Use EnsembleRetriever?

| Approach | Implementation | Use Case |
|----------|---------------|----------|
| **Qdrant native hybrid** | Requires Qdrant-specific code | When you control the vector DB |
| **LangChain EnsembleRetriever** | Works with ANY retrievers | **Production standard** |

> **EnsembleRetriever is framework-agnostic** - combine ANY retrievers (BM25, Qdrant, Pinecone, etc.)

In [58]:
# Step 1: Create individual retrievers
# Reference: https://python.langchain.com/docs/modules/data_connection/retrievers/

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever  # CORRECT: from langchain, not langchain_community

print("Building LangChain hybrid search system...\n")

# ============================================================
# RETRIEVER 1: BM25 (Sparse - Keyword Search)
# ============================================================
# BM25 is a sparse retriever that finds exact keyword matches
# Official docs: https://python.langchain.com/docs/integrations/retrievers/bm25/

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)
# Set the number of documents to retrieve
bm25_retriever.k = 3

print("1. BM25Retriever created")
print("   - Algorithm: BM25 (Best Match 25)")
print("   - Type: Sparse/Keyword search")
print("   - k=3 (returns top 3 documents)")
print("   - Best for: Exact matches, product codes, IDs\n")

# ============================================================
# RETRIEVER 2: Qdrant (Dense - Semantic Search)
# ============================================================
# Qdrant retriever uses embeddings for semantic similarity
# Official docs: https://python.langchain.com/docs/integrations/vectorstores/qdrant/

qdrant_retriever = dense_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("2. Qdrant Retriever created")
print("   - Embedding: all-MiniLM-L6-v2 (384 dims)")
print("   - Type: Dense/Semantic search")
print("   - k=3 (returns top 3 documents)")
print("   - Best for: Natural language, meaning, synonyms\n")

# ============================================================
# RETRIEVER 3: EnsembleRetriever (Hybrid)
# ============================================================
# Combines multiple retrievers using Reciprocal Rank Fusion (RRF)
# Official docs: https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble/

hybrid_retriever = EnsembleRetriever(
    retrievers=[qdrant_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # Equal weight: 50% dense, 50% sparse
)

print("3. EnsembleRetriever created (HYBRID)")
print("   - Fusion method: Reciprocal Rank Fusion (RRF)")
print("   - Weights: [0.5, 0.5] (equal contribution)")
print("   - Combines: Dense semantic + Sparse keyword search")
print("   - Best for: ALL query types (production standard)")
print("\nHybrid retriever ready for testing!")

ModuleNotFoundError: No module named 'langchain.retrievers'

In [ ]:
# Compare all three retrievers on different query types
# Using .invoke() method (LangChain LCEL standard)

print("=" * 70)
print("TEST 1: Exact Chemical Name Query")
print("=" * 70)
query1 = "3-fluorophenmetrazine"
print(f"Query: '{query1}'")
print("Expected: Sparse (BM25) should excel at exact matching\n")

# Invoke all three retrievers
dense_results = qdrant_retriever.invoke(query1)
sparse_results = bm25_retriever.invoke(query1)
hybrid_results = hybrid_retriever.invoke(query1)

print("Dense (Qdrant) - Top result:")
has_match = query1.lower() in dense_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {dense_results[0].page_content[:120]}...\n")

print("Sparse (BM25) - Top result:")
has_match = query1.lower() in sparse_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {sparse_results[0].page_content[:120]}...\n")

print("Hybrid (Ensemble) - Top result:")
has_match = query1.lower() in hybrid_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {hybrid_results[0].page_content[:120]}...\n")

print("=" * 70)
print("TEST 2: Semantic Query (Natural Language)")
print("=" * 70)
query2 = "Which substances cause opioid-like effects?"
print(f"Query: '{query2}'")
print("Expected: Dense should excel at understanding meaning\n")

dense_results2 = qdrant_retriever.invoke(query2)
sparse_results2 = bm25_retriever.invoke(query2)
hybrid_results2 = hybrid_retriever.invoke(query2)

print("Dense (Qdrant) - Top result:")
print(f"  {dense_results2[0].page_content[:120]}...\n")

print("Sparse (BM25) - Top result:")
print(f"  {sparse_results2[0].page_content[:120]}...\n")

print("Hybrid (Ensemble) - Top result:")
print(f"  {hybrid_results2[0].page_content[:120]}...\n")

print("=" * 70)
print("CONCLUSION: Why Use EnsembleRetriever?")
print("=" * 70)
print("The Hybrid retriever (EnsembleRetriever) combines strengths:")
print("  1. Finds exact matches (like BM25)")
print("  2. Understands meaning (like dense search)")
print("  3. Works with ANY LangChain retriever")
print("  4. Production-ready with automatic RRF fusion")
print("\nThis is the recommended approach for production RAG systems!")

---

## 7. Key Takeaways

### Dense vs Sparse Vectors

| Feature | Dense (Embeddings) | Sparse (Keywords) | Hybrid (Both) |
|---------|-------------------|------------------|---------------|
| **Best for** | Meaning, synonyms | Exact matches | All query types |
| **Example queries** | "heart attack symptoms" | "3-fluorophenmetrazine" | Both! |
| **Vector size** | Fixed (384-1536 dims) | Variable (only non-zero words) | Both |
| **Storage** | Dense arrays | Sparse indices + values | Both |
| **Production use** | Good | Good | Best! |

### Two Approaches to Hybrid Search

| Approach | Implementation | Pros | Cons |
|----------|---------------|------|------|
| **Qdrant Native** | Store dense + sparse in same collection | Fast, single DB query | Vendor-specific code |
| **LangChain EnsembleRetriever** | Combine any retrievers with RRF | Framework-agnostic, flexible | Two separate queries |

### When to Use Each Approach

**Dense-only (embeddings):**
- Natural language questions
- Semantic search
- No exact product codes or IDs
- Example: Customer support chatbot

**Sparse-only (keywords):**
- Exact code lookup
- Product catalog search
- Legal document search (exact citations)
- Example: Product manual search

**Hybrid (dense + sparse):**
- Technical documentation (codes + descriptions)
- Medical/pharmaceutical databases (exact drug names + symptoms)
- E-commerce (product codes + natural queries)
- **Recommended for all production RAG systems**

### Implementation Patterns

**Pattern 1: Dense-only (Simple)**
```python
from langchain_qdrant import QdrantVectorStore

dense_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_collection",
)
retriever = dense_store.as_retriever(search_kwargs={"k": 3})
```

**Pattern 2: LangChain Hybrid (RECOMMENDED)**
```python
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Create individual retrievers
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

dense_retriever = dense_store.as_retriever(search_kwargs={"k": 3})

# Combine with RRF
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

# Use in RAG pipeline
results = hybrid_retriever.invoke("your query")
```

**Pattern 3: Qdrant Native Hybrid (Advanced)**
```python
from qdrant_client import QdrantClient, models

# Create collection with both vector types
client.create_collection(
    collection_name="hybrid_collection",
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams(...)},
)

# Search using specific vector type
results = client.query_points(
    collection_name="hybrid_collection",
    query=vector,
    using="dense",  # or "sparse"
    limit=3,
)
```

### Production Recommendations

1. **Start with EnsembleRetriever** - Works with any LangChain-compatible retriever
2. **Use equal weights [0.5, 0.5]** - Adjust based on your specific use case
3. **Monitor performance** - Track which retriever contributes most to good results
4. **Consider Qdrant native** - If you need maximum performance and control

> **For most production RAG systems, use LangChain's EnsembleRetriever** for flexibility and ease of use.

---

## Appendix: PEP 8 Style Guide

This notebook follows Python PEP 8 style conventions:

### Naming Conventions

- **Variables/functions:** `snake_case` (e.g., `dense_store`, `create_sparse_vector`)
- **Constants:** `UPPER_CASE` (e.g., `QDRANT_URL`, `QDRANT_API_KEY`)
- **Classes:** `PascalCase` (e.g., `QdrantClient`, `VectorParams`)

### Code Formatting

- Spaces around operators: `size = 384` not `size=384`
- Space after commas: `[1, 2, 3]` not `[1,2,3]`
- Indentation: 4 spaces (not tabs)
- Line length: < 100 characters where possible
- Docstrings: Triple quotes for functions

### Best Practices

- Use f-strings for formatting: `f"Result: {score}"` 
- Descriptive variable names: `dense_vector` not `dv`
- Comments explain WHY, not WHAT
- One import per line

---

*Hope to Skill — Building the future, one skill at a time.*